## Imports

In [2]:
import os
import pandas as pd
from PIL import Image
from fractions import Fraction
import shutil

# === CONFIG ===
# Hardcode your main input folder (adjust as needed)
INPUT_FOLDER = "/Users/matt/Desktop/TeedUp/Code/Mockups/script testing/graphic+text"

# Paths derived from INPUT_FOLDER
CSV_OUTPUT   = os.path.join(INPUT_FOLDER, "aspect_ratios.csv")
BY_RATIO_DIR = os.path.join(INPUT_FOLDER, "_by_ratio")

# Folder naming for per-ratio buckets
INPUT_SUFFIX  = "_input"   # where originals get copied for PS batching
OUTPUT_SUFFIX = "_output"  # where PS scripts should export final overlays

# Ensure parent output exists
os.makedirs(BY_RATIO_DIR, exist_ok=True)

print(f"Analyzing designs in: {INPUT_FOLDER}")
print(f"Per-ratio folders will be created under: {BY_RATIO_DIR}")
print(f"Suffixes -> input='{INPUT_SUFFIX}', output='{OUTPUT_SUFFIX}'")


Analyzing designs in: /Users/matt/Desktop/TeedUp/Code/Mockups/script testing/graphic+text
Per-ratio folders will be created under: /Users/matt/Desktop/TeedUp/Code/Mockups/script testing/graphic+text/_by_ratio
Suffixes -> input='_input', output='_output'


## CSV Analysis Function

In [3]:
def get_aspect_ratio(path, tolerance=0.05):
    """Return simplified aspect ratio string like '1x1' or '3x4'."""
    with Image.open(path) as img:
        w, h = img.size
    ratio = w / h

    # Common standard aspect ratios
    common_ratios = {
        "1x1": 1.0,
        "3x4": 0.75,
        "4x5": 0.8,
        "2x3": 0.67,
        "9x16": 0.5625,
        "5x4": 1.25,
        "16x9": 1.78,
    }

    for name, val in common_ratios.items():
        if abs(ratio - val) < tolerance:
            return name

    # Fallback: auto simplify ratio if not common
    frac = Fraction(ratio).limit_denominator(10)
    return f"{frac.numerator}x{frac.denominator}"


## Generate CSV

In [4]:
data = []

for root, _, files in os.walk(INPUT_FOLDER):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
            full_path = os.path.join(root, f)
            ar = get_aspect_ratio(full_path)
            data.append({
                "filename": f,
                "path": full_path,
                "aspect_ratio": ar
            })

df = pd.DataFrame(data)
df.to_csv(CSV_OUTPUT, index=False)
print(f"✅ CSV saved to: {CSV_OUTPUT}")
df.head()


✅ CSV saved to: /Users/matt/Desktop/TeedUp/Code/Mockups/script testing/graphic+text/aspect_ratios.csv


,filename,path,aspect_ratio
0,my_five_iron.png,/Users/matt/Desktop/TeedUp/Code/Mockups/script...,2x3
1,catty_caddy.png,/Users/matt/Desktop/TeedUp/Code/Mockups/script...,3x4
2,low_handicap_high_standards_green.png,/Users/matt/Desktop/TeedUp/Code/Mockups/script...,1x1
3,shes_watching_the_kids_green.png,/Users/matt/Desktop/TeedUp/Code/Mockups/script...,1x1
4,bogeyman.png,/Users/matt/Desktop/TeedUp/Code/Mockups/script...,3x4


## Sort subfolder by Ratio

In [5]:
# df must already exist from Cell 3
if 'df' not in globals():
    raise RuntimeError("DataFrame 'df' not found. Run Cells 1–3 first.")

total = 0
for _, row in df.iterrows():
    ratio = row["aspect_ratio"]

    r_input_dir  = os.path.join(BY_RATIO_DIR, f"{ratio}{INPUT_SUFFIX}")
    r_output_dir = os.path.join(BY_RATIO_DIR, f"{ratio}{OUTPUT_SUFFIX}")  # created now, filled later by PS

    os.makedirs(r_input_dir,  exist_ok=True)
    os.makedirs(r_output_dir, exist_ok=True)

    src = row["path"]
    dst = os.path.join(r_input_dir, row["filename"])
    shutil.copy2(src, dst)  # switch to shutil.move if you prefer to relocate
    total += 1

print(f"✅ Copied {total} files into per-ratio '*{INPUT_SUFFIX}' folders.")
print(f"✅ Created matching '*{OUTPUT_SUFFIX}' folders for Photoshop exports.")


✅ Copied 39 files into per-ratio '*_input' folders.
✅ Created matching '*_output' folders for Photoshop exports.


### From here, run the Design_Overlay scripts for each of the respective ratios

## Regeneration Logic
Run this cell after the Photoshop script has each image as a mockup image

In [ ]:
"""
REINTEGRATION LOGIC

Reads the CSV created in Cell 3, then for each row:
- Looks for the processed file in '<ratio>_output' under BY_RATIO_DIR
- Copies (or moves) it back to the file's original directory from the CSV
"""

REINTEGRATION_MODE = "copy"  # "copy" or "move"

# Load aspect ratio mapping
csv_path = os.path.join(INPUT_FOLDER, "aspect_ratios.csv")
if not os.path.exists(csv_path):
    raise FileNotFoundError("aspect_ratios.csv not found — run Cells 1–4 first")
df_idx = pd.read_csv(csv_path)

moved_back = 0
missing = 0

for _, row in df_idx.iterrows():
    ratio = row["aspect_ratio"]
    filename = row["filename"]
    original_dir = os.path.dirname(row["path"])

    # where Photoshop should have exported to
    src = os.path.join(BY_RATIO_DIR, f"{ratio}{OUTPUT_SUFFIX}", filename)
    dst = os.path.join(original_dir, filename)

    if os.path.exists(src):
        os.makedirs(original_dir, exist_ok=True)
        if REINTEGRATION_MODE == "move":
            if os.path.exists(dst): os.remove(dst)
            shutil.move(src, dst)
        else:
            shutil.copy2(src, dst)
        moved_back += 1
    else:
        missing += 1
        print(f"⚠️ Missing processed file: {src}")

print(f"✅ Reintegrated {moved_back} files to their original folders.")
if missing:
    print(f"⚠️ {missing} files were not found in '*{OUTPUT_SUFFIX}' folders.")
